<a href="https://colab.research.google.com/github/tadeugomes/2_encontro_enap/blob/main/Aula5_Exercicio5_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercício 5.1 - Solução

In [1]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')

Authenticated


In [4]:
import pandas as pd
## Defina o id do seu projeto no bigquery!!!!!
project_id = 'enap-mba-470912' # Defina o id do seu projeto no bigquery!!!!!
## Defina o id do seu projeto no bigquery!!!!!

df = pd.io.gbq.read_gbq('''
SELECT
  pib.ano AS ano,
  uf.sigla AS sigla_uf,
  mun.id_municipio,
  pop.populacao,
  mun.nome AS nome_municipio,
  pib.pib,
  ROUND(CAST(pib.pib AS FLOAT64)/NULLIF(CAST(pop.populacao AS FLOAT64),0), 6) AS pibpercapita
FROM
  `basedosdados.br_ibge_pib.municipio` AS pib
JOIN
  `basedosdados.br_ibge_populacao.municipio` AS pop
    ON pib.id_municipio = pop.id_municipio AND pib.ano = pop.ano
JOIN
  `basedosdados.br_bd_diretorios_brasil.municipio` AS mun
    ON pib.id_municipio = mun.id_municipio
JOIN
  `basedosdados.br_bd_diretorios_brasil.uf` AS uf
    ON mun.id_uf = uf.id_uf
WHERE
  pib.ano BETWEEN 2002 AND 2018
ORDER BY
  ano, sigla_uf, nome_municipio
''', project_id=project_id)

df.head()

/tmp/ipython-input-679519711.py:6: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df = pd.io.gbq.read_gbq('''


,ano,sigla_uf,id_municipio,populacao,nome_municipio,pib,pibpercapita
0,2002,AC,1200013,8454,Acrelândia,39622000,4686.775491
1,2002,AC,1200054,3611,Assis Brasil,12789000,3541.678205
2,2002,AC,1200104,17649,Brasiléia,72997000,4136.041702
3,2002,AC,1200138,6382,Bujari,26743000,4190.379191
4,2002,AC,1200179,5814,Capixaba,32492000,5588.579291


In [25]:
import pandas as pd

# Defina o id do seu projeto no BigQuery
project_id = "enap-mba-470912"

query = """
WITH
  pib_data AS (
    SELECT ano, id_municipio, pib
    FROM `basedosdados.br_ibge_pib.municipio`
    WHERE ano BETWEEN 2002 AND 2018
  ),
  pop_data AS (
    SELECT ano, id_municipio, populacao
    FROM `basedosdados.br_ibge_populacao.municipio`
    WHERE ano BETWEEN 2002 AND 2018
  ),
  mun_data AS (
    SELECT id_municipio, nome, id_uf
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
  ),
  uf_data AS (
    SELECT id_uf, sigla
    FROM `basedosdados.br_bd_diretorios_brasil.uf`
  ),

  -- 1) Base “longa” apenas com o que interessa para pivotar
  saeb_base AS (
    SELECT
      id_municipio,
      ano,
      rede,
      nota_saeb_media_padronizada
    FROM `basedosdados.br_inep_ideb.municipio`
    WHERE rede IN ('municipal','estadual','federal','publica')
  ),

  -- 2) Pivot: redes viram colunas (evita replicar PIB/POP por escola/rede)
  saeb_pivot AS (
    SELECT
      id_municipio,
      ano,
      municipal AS saeb_municipal,
      estadual  AS saeb_estadual,
      federal   AS saeb_federal,
      publica   AS saeb_publica
    FROM saeb_base
    PIVOT (
      MAX(nota_saeb_media_padronizada) FOR rede IN ('municipal','estadual','federal','publica')
    )
  )

-- 3) Join final com o modelo PIB + Pop + UF/Município
SELECT
  p.ano AS ano,
  u.sigla AS sigla_uf,
  m.id_municipio,
  pop.populacao,
  m.nome AS nome_municipio,
  p.pib,
  ROUND(CAST(p.pib AS FLOAT64) / NULLIF(CAST(pop.populacao AS FLOAT64), 0), 6) AS pibpercapita,
  s.saeb_municipal,
  s.saeb_estadual,
  s.saeb_federal,
  s.saeb_publica
FROM pib_data p
JOIN pop_data pop
  ON p.id_municipio = pop.id_municipio AND p.ano = pop.ano
JOIN mun_data m
  ON p.id_municipio = m.id_municipio
JOIN uf_data u
  ON m.id_uf = u.id_uf
LEFT JOIN saeb_pivot s
  ON p.id_municipio = s.id_municipio AND p.ano = s.ano
ORDER BY p.ano, sigla_uf, nome_municipio
"""

df_saeb = pd.io.gbq.read_gbq(query, project_id=project_id)
df_saeb.head()

/tmp/ipython-input-2256581409.py:94: FutureWarning: read_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.read_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.read_gbq
  df_saeb = pd.io.gbq.read_gbq(query, project_id=project_id)


GenericGBQException: Reason: 400 POST https://bigquery.googleapis.com/bigquery/v2/projects/enap-mba-470912/queries?prettyPrint=false: Unrecognized name: `1_mescrita`; Did you mean mescrita_1? at [59:5]

In [22]:
# Check the number of rows before dropping NaN values
print(f"Shape of the DataFrame before dropping NaNs: {df_saeb.shape}")

# Drop rows where any of the saeb columns are NaN
saeb_cols = ['saeb_municipal', 'saeb_estadual', 'saeb_federal', 'saeb_publica']
df_saeb_cleaned = df_saeb.dropna(subset=saeb_cols)

# Check the number of rows after dropping NaN values
print(f"Shape of the DataFrame after dropping NaNs in SAEB columns: {df_saeb_cleaned.shape}")

# Display the head of the cleaned DataFrame
display(df_saeb_cleaned.head())

Shape of the DataFrame before dropping NaNs: (94625, 11)
Shape of the DataFrame after dropping NaNs in SAEB columns: (344, 11)


,ano,sigla_uf,id_municipio,populacao,nome_municipio,pib,pibpercapita,saeb_municipal,saeb_estadual,saeb_federal,saeb_publica
27827,2007,AC,1200401,290639,Rio Branco,3406220000,11719.762317,4.859643,4.744907,5.771516,4.775920
28347,2007,BA,2927408,2892625,Salvador,28063726000,9701.819628,4.485481,4.451641,7.547833,4.479346
28489,2007,CE,2304400,2431415,Fortaleza,24042584000,9888.309482,4.387938,4.497515,7.444500,4.412242
28788,2007,GO,5208707,1244645,Goiânia,20755041000,16675.470516,4.525262,4.694101,6.261000,4.566899
29130,2007,MA,2111300,957515,São Luís,11395641000,11901.266299,4.513563,4.556088,4.710450,4.534817


## [Demonstração](https://drive.google.com/file/d/1PYtx7LSVGYzv3pcjPY6VB8uO7P1Cl0cF/view?usp=sharing) de como criar um dataset

## Submeta o seu dataframe criando uma tabela no BigQuery

In [ ]:
# Enviar o DataFrame para o BigQuery
try:
    df.to_gbq(
        "enapcd2021.pibpercapita",
        project_id=project_id,
        chunksize=40000,
        if_exists='replace'
    )
    print("Dados enviados com sucesso para a tabela 'enapcd2021.pibpercapita'")
except Exception as e:
    print(f"Erro ao enviar dados para o BigQuery: {e}")
    print("\nVerifique:")
    print("1. Se o dataset 'enapcd2021' existe no seu projeto")
    print("2. Se você tem permissão para criar tabelas")
    print("3. Se o nome da tabela está correto")
    print("4. Sua autenticação está funcionando corretamente")